In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch.nn as nn

def parse_catalogcontent(text):
    if pd.isna(text):
        return {}, text

    item_name_match = re.search(r'Item Name\s*:\s*(.+)', text, re.I)
    if not item_name_match:
        item_name = text.split('\n')[0].strip()
    else:
        item_name = item_name_match.group(1).split('\n')[0].strip()
    bullet_points = ".".join(re.findall(r'Bullet Point[\s\d]*:\s*(.+?)(?:\n|$)', text, re.I))
    prod_desc_match = re.search(r'Product Description\s*:\s*(.+)', text, re.I | re.S)
    prod_desc = prod_desc_match.group(1).strip() if prod_desc_match else ""
    value_match = re.search(r'([\d.,]+)\s*(kg|g|mg|lb|oz|l|ml|count|bottle|pack|box|piece|jar|tin|case|bag|packet|sachet|unit|m|cm|in|ft)?', text, re.I)
    valuenumeric = None
    unit = None
    if value_match:
        vn = value_match.group(1).replace(',', '')
        try:
            valuenumeric = float(vn)
        except:
            valuenumeric = None
        unit = value_match.group(2).lower() if value_match.group(2) else None
    combo_pattern = r'pack|combo|set of|bundle|value pack|multi pack|2-in-1|3 pack|pack of|qty'
    combo = 1 if re.search(combo_pattern, text, re.I) else 0
    brand = item_name.split()[0] if item_name else ""
    bullet_point_count = text.lower().count('bullet point')
    text_length = len(text)
    has_description = 1 if prod_desc else 0
    numeric_value_missing = 1 if valuenumeric is None else 0

    return {
        'ItemName': item_name,
        'BulletPoints': bullet_points,
        'ProductDescription': prod_desc,
        'ValueNumeric': valuenumeric if valuenumeric is not None else 0,
        'Unit': unit if unit else 'count',
        'Combo': combo,
        'Brand': brand,
        'BulletPointCount': bullet_point_count,
        'TextLength': text_length,
        'HasDescription': has_description,
        'NumericValueMissing': numeric_value_missing
    }, text

class PriceDataset(Dataset):
    def __init__(self, texts, targets, tokenizer, max_length=512):
        self.texts = texts
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        target = self.targets[idx]
        enc = self.tokenizer(text, padding='max_length', truncation=True, max_length=self.max_length, return_tensors='pt')
        item = {k: v.squeeze(0) for k,v in enc.items()}
        item['labels'] = torch.tensor(target, dtype=torch.float32)
        return item

class QwenRegressionModel(torch.nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        hidden_size = base_model.config.hidden_size
        self.regressor = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=False, return_dict=True)
        last_hidden = outputs.last_hidden_state
        last_token_hidden = last_hidden[:, -1, :]
        regression_output = self.regressor(last_token_hidden).squeeze(-1)
        loss = None
        if labels is not None:
            loss_fn = nn.MSELoss()
            loss = loss_fn(regression_output, labels)
        return {'loss': loss, 'logits': regression_output}

def compute_metrics(preds, labels):
    preds_exp = np.expm1(preds)
    labels_exp = np.expm1(labels)
    n = len(preds_exp)
    smape = 100/n * np.sum(2 * np.abs(preds_exp - labels_exp) / (np.abs(preds_exp) + np.abs(labels_exp) + 1e-8))
    mape = 100 * np.mean(np.abs((labels_exp - preds_exp) / (labels_exp + 1e-8)))
    mae = mean_absolute_error(labels_exp, preds_exp)
    mse = mean_squared_error(labels_exp, preds_exp)
    rmse = np.sqrt(mse)
    r2 = r2_score(labels_exp, preds_exp)
    return {'SMAPE': smape, 'MAPE': mape, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

def main():
    # Global file paths
    TRAIN_CSV_PATH = '/Users/gabi/Desktop/DL stuff/AMAZONML/student_resource/dataset/train.csv'
    TEST_CSV_PATH = '/Users/gabi/Desktop/DL stuff/AMAZONML/student_resource/dataset/test.csv'
    OUTPUT_CSV_PATH = 'test_out.csv'
    MODEL_NAME = "Qwen/Qwen-2.5-2B-Instruct"
    
    df = pd.read_csv(TRAIN_CSV_PATH)
    parsed_cols = df['catalogcontent'].apply(lambda x: parse_catalogcontent(x)[0])
    parsed_df = pd.DataFrame(parsed_cols.tolist())
    df = pd.concat([df, parsed_df], axis=1)

    unit_map = {
        'oz': 'oz', 'ounce': 'oz', 'ounces': 'oz',
        'lb': 'lb', 'pound': 'lb', 'pounds': 'lb', 'lbs': 'lb',
        'g': 'g', 'gram': 'g', 'grams': 'g', 'gr': 'g',
        'kg': 'kg', 'kilogram': 'kg', 'kilograms': 'kg',
        'l': 'l', 'liter': 'l', 'liters': 'l', 'ltr': 'l',
        'ml': 'ml', 'milliliter': 'ml', 'milliliters': 'ml', 'mls': 'ml',
        'count': 'count',
        'pack': 'pack',
    }
    df['UnitNorm'] = df['Unit'].apply(lambda u: unit_map.get(u.lower(), 'count') if isinstance(u, str) else 'count')

    brand_map = {b: i for i, b in enumerate(df['Brand'].unique())}
    df['BrandEnc'] = df['Brand'].map(brand_map)

    unit_inv_map = {u:i for i,u in enumerate(set(df['UnitNorm']))}
    df['UnitEnc'] = df['UnitNorm'].map(unit_inv_map)

    df['ValueNumeric'].fillna(0, inplace=True)

    feature_cols = ['ValueNumeric', 'Combo', 'BrandEnc', 'BulletPointCount', 'TextLength', 'HasDescription', 'NumericValueMissing', 'UnitEnc']
    X = df[feature_cols].fillna(0).values

    num_clusters = 10
    kmeans = KMeans(n_clusters=num_clusters, random_state=42)
    clusters = kmeans.fit_predict(X)
    df['Cluster'] = clusters

    samples_per_cluster = 100
    sample_indices = []
    for cl in range(num_clusters):
        cl_idx = df[df['Cluster'] == cl].index
        if len(cl_idx) > samples_per_cluster:
            sampled = np.random.choice(cl_idx, samples_per_cluster, replace=False)
        else:
            sampled = cl_idx
        sample_indices.extend(sampled)

    df_sampled = df.loc[sample_indices].reset_index(drop=True)

    df_sampled['PriceLog1p'] = np.log1p(df_sampled['price'])

    train_df, val_df = train_test_split(df_sampled, test_size=0.2, random_state=42)

    def format_prompt(row):
        numeric_str = " ".join([
            f"Value:{row['ValueNumeric']}",
            f"Combo:{row['Combo']}",
            f"BrandEnc:{row['BrandEnc']}",
            f"BulletPointsCount:{row['BulletPointCount']}",
            f"TextLength:{row['TextLength']}",
            f"HasDescription:{row['HasDescription']}",
            f"NumericValueMissing:{row['NumericValueMissing']}",
            f"UnitEnc:{row['UnitEnc']}"
        ])
        return row['catalogcontent'] + " " + numeric_str

    train_texts = train_df.apply(format_prompt, axis=1).tolist()
    train_labels = train_df['PriceLog1p'].values

    val_texts = val_df.apply(format_prompt, axis=1).tolist()
    val_labels = val_df['PriceLog1p'].values

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True)

    reg_model = QwenRegressionModel(base_model)

    train_dataset = PriceDataset(train_texts, train_labels, tokenizer)
    val_dataset = PriceDataset(val_texts, val_labels, tokenizer)

    training_args = TrainingArguments(
        output_dir='./results',
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        num_train_epochs=3,
        weight_decay=0.01,
        learning_rate=3e-5,
        logging_dir='./logs',
        logging_steps=20,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
    )

    class RegressionTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False):
            labels = inputs.get("labels")
            outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"], labels=labels)
            loss = outputs["loss"]
            return (loss, outputs) if return_outputs else loss
        
        def evaluation_step(self, model, inputs):
            with torch.no_grad():
                outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"], labels=inputs.get("labels"))
                preds = outputs['logits'].detach().cpu().numpy()
                labels = inputs.get("labels").detach().cpu().numpy()
            return preds, labels

        def evaluate(self, eval_dataset=None):
            eval_dataset = eval_dataset if eval_dataset is not None else self.eval_dataset
            preds, labels = [], []
            for i in range(len(eval_dataset)):
                inputs = eval_dataset[i]
                inputs = {k: v.unsqueeze(0).to(self.model.device) for k, v in inputs.items()}
                pred, label = self.evaluation_step(self.model, inputs)
                preds.append(pred[0])
                labels.append(label[0])
            metrics = compute_metrics(np.array(preds), np.array(labels))
            print("Validation metrics:", metrics)
            return metrics

    def predict_and_save(test_path=TEST_CSV_PATH, out_path=OUTPUT_CSV_PATH):
        test_df = pd.read_csv(test_path)
        parsed_cols = test_df['catalogcontent'].apply(lambda x: parse_catalogcontent(x)[0])
        parsed_df = pd.DataFrame(parsed_cols.tolist())
        test_df_ext = pd.concat([test_df, parsed_df], axis=1)

        test_df_ext['UnitNorm'] = test_df_ext['Unit'].apply(lambda u: unit_map.get(u.lower(), 'count') if isinstance(u, str) else 'count')
        test_df_ext['BrandEnc'] = test_df_ext['Brand'].map(brand_map).fillna(-1).astype(int)
        test_df_ext['UnitEnc'] = test_df_ext['UnitNorm'].map(unit_inv_map).fillna(unit_inv_map['count']).astype(int)
        test_df_ext['ValueNumeric'].fillna(0, inplace=True)

        def format_prompt_test(row):
            numeric_str = " ".join([
                f"Value:{row['ValueNumeric']}",
                f"Combo:{row['Combo']}",
                f"BrandEnc:{row['BrandEnc']}",
                f"BulletPointsCount:{row['BulletPointCount']}",
                f"TextLength:{row['TextLength']}",
                f"HasDescription:{row['HasDescription']}",
                f"NumericValueMissing:{row['NumericValueMissing']}",
                f"UnitEnc:{row['UnitEnc']}"
            ])
            return row['catalogcontent'] + " " + numeric_str

        texts = test_df_ext.apply(format_prompt_test, axis=1).tolist()

        reg_model.eval()
        prices_pred = []
        with torch.no_grad():
            for text in texts:
                inputs = tokenizer(text, padding='max_length', truncation=True, max_length=512, return_tensors='pt').to(reg_model.base_model.device)
                outputs = reg_model(**inputs)
                pred_log = outputs['logits'].cpu().item()
                pred = np.expm1(pred_log)
                prices_pred.append(pred)

        out_df = pd.DataFrame({'sample_id': test_df_ext['sampleid'], 'price': prices_pred})
        out_df.to_csv(out_path, index=False)
        print(f"Test predictions saved to {out_path}")

    trainer = RegressionTrainer(
        model=reg_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset
    )

    trainer.train()

    metrics = trainer.evaluate()
    print("\nFinal Validation Metrics:")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

    predict_and_save()

if __name__ == "__main__":
    main()


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
